# 📄 NeSy-DocAI: Neuro-Symbolic Document AI Research Notebook

**NeSy-DocAI** is a research framework for zero-hallucination financial invoice parsing combining:
- **System 1 (Visual Perception Engine)**: Multimodal Vision-Language Model (`Qwen2.5-VL` / OCR) extracting 2D bounding boxes and OCR candidate lattices.
- **System 2 (Symbolic Reasoning Engine)**: **Z3 SMT Constraint Solver** enforcing Presburger integer accounting equations and correcting OCR character confusion errors (`O` -> `0`, `l` -> `1`, `S` -> `5`).


In [ ]:
import sys
import json
from PIL import Image

# Import nesy-docai modules
from nesy_docai import (
    VisionPerceptionEngine,
    SymbolicSolverEngine,
    TaxMasterDataVerifier,
    AuditExcelExporter,
    BoundingBoxVisualizer,
    PDFDocumentProcessor
)

print("✅ nesy-docai framework modules loaded successfully!")

## 👁️ Step 1: System 1 Visual Perception (Candidate Extractions)
Raw extractions from visual OCR models frequently suffer from character confusion noise (e.g. `'1O000'` instead of `10000`, `'95OO'` instead of `9500`).

In [ ]:
vision = VisionPerceptionEngine()
raw_data = vision.process_invoice_image("sample_invoice.png")

print("📥 Raw OCR Output (Contains digit noise '1O000' and '95OO'):")
print(json.dumps(raw_data, ensure_ascii=False, indent=2))

## 🧠 Step 2: System 2 Symbolic Reasoning (Z3 Presburger SMT Solver)
The Z3 Solver enforces hard accounting constraints:
- Line Amount = Quantity * Unit Price
- Subtotal = Sum(Line Amounts)
- Total = Subtotal + Tax

It resolves OCR character confusion in ~2.8 milliseconds and outputs an auditable SAT proof certificate.

In [ ]:
solver = SymbolicSolverEngine(vision_engine=vision)
verified_record = solver.solve_and_verify(raw_data)

print(f"⚙️ SMT Solver Status: {verified_record['audit_status']}")
print("✨ Corrected & Verified Financial Record:")
print(json.dumps(verified_record, ensure_ascii=False, indent=2))

## 🏛️ Step 3: Tax Master Data & Legal Verification

In [ ]:
tax_verifier = TaxMasterDataVerifier()
tax_info = tax_verifier.verify_tax_id(verified_record['seller_tax_id'])
verified_record['tax_verification'] = tax_info

print("🏛️ General Department of Taxation Cross-Check:")
print(json.dumps(tax_info, ensure_ascii=False, indent=2))

## 🖼️ Step 4: Pixel-Level Provenance & Bounding Box Annotation

In [ ]:
visualizer = BoundingBoxVisualizer()
pdf_proc = PDFDocumentProcessor()

# Create sample invoice page image
invoice_img = pdf_proc.render_dummy_pdf_page_image(page_num=1)
annotated_img = visualizer.annotate_invoice(invoice_img, verified_record)

# Display annotated image
annotated_img.save("annotated_sample_invoice.png")
display(annotated_img)

## 📊 Step 5: Multi-Sheet Excel Audit Ledger Export

In [ ]:
exporter = AuditExcelExporter()
excel_path = exporter.export_to_excel([verified_record], output_filepath="invoice_audit_report.xlsx")

print(f"💾 Excel Audit Report saved to: {excel_path}")